# 00 — Colab: Prepare Data and Kaggle Input

Run this notebook in **Google Colab**. It creates the deterministic Parquet datasets, links durable directories to Google Drive, and exports a private Kaggle input bundle. Start with the `smoke` profile.

In [ ]:
REPO_URL = "https://github.com/MichealSK/political-bias-lab.git"
PROFILE = "smoke"  # smoke -> pilot -> paper
DRIVE_ROOT = "/content/drive/MyDrive/political-bias-lab"
REPO_DIR = "/content/political-bias-lab"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])

## Link persistent directories to Drive

`prepared/`, `results/`, and `paper_bundle/` will live on Drive. The Git clone remains disposable.

In [ ]:
from src.cloud import link_colab_persistent_dirs
link_colab_persistent_dirs(REPO_DIR, DRIVE_ROOT)
print("Persistent root:", DRIVE_ROOT)

## Optional: real-politician study

For a paper that claims bias toward real politicians, place `entities_research.csv` and `entity_pairs_research.csv` in `data/` **before** this cell. If they are absent, the benchmark uses 500 deterministic fictional names and should be interpreted as entity/name sensitivity.

In [ ]:
from pathlib import Path
from src.config import load_config
from src.pipeline import prepare_project
from src.reproducibility import save_run_manifest

cfg = load_config(Path(REPO_DIR)/"config/default.yaml", Path(REPO_DIR)/f"config/{PROFILE}.yaml")
stats = prepare_project(cfg, root=REPO_DIR)
save_run_manifest(Path(REPO_DIR)/"results/manifests/colab_prepare_manifest.json", config=cfg, root=REPO_DIR, extra={"stage":"colab_prepare"})
stats

In [ ]:
# Validate the logical universe.
expected = cfg["phase1"]["logical_universe"]["expected_cross_product"]
if not stats["uses_real_entities"]:
    assert stats["logical_swap_cases"] == expected
print(f"Logical swap universe: {stats['logical_swap_cases']:,}")
print(f"Materialized Phase-1 paired rows before A/B expansion: {stats['phase1_sample_rows']:,}")

## Export the Kaggle input bundle

In [ ]:
from src.transfer import make_kaggle_input_bundle
from pathlib import Path
import shutil

local_bundle = Path(REPO_DIR)/"transfer"/f"kaggle_input_{PROFILE}.zip"
make_kaggle_input_bundle(REPO_DIR, local_bundle)
drive_transfer = Path(DRIVE_ROOT)/"transfer"
drive_transfer.mkdir(parents=True, exist_ok=True)
drive_bundle = drive_transfer/local_bundle.name
shutil.copy2(local_bundle, drive_bundle)
print("Kaggle input bundle:", drive_bundle)

### Next

1. In Kaggle, create a **private Dataset** from the generated ZIP (or extracted `prepared/*.parquet`).
2. Open `01_kaggle_inference.ipynb`.
3. Enable GPU + Internet.
4. Attach the private Dataset.
5. Use the **same profile** you used here.